## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Results directory
results_dir = Path('results')

print("Loading results...")

In [ ]:
# Load all results
wachter_df = pd.read_csv(results_dir / 'wachter_results.csv')
growing_spheres_df = pd.read_csv(results_dir / 'growing_spheres_results.csv')
prototype_df = pd.read_csv(results_dir / 'prototype_results.csv')
gradient_df = pd.read_csv(results_dir / 'gradient_based_results.csv')

print(f"Wachter: {len(wachter_df)} results")
print(f"Growing Spheres: {len(growing_spheres_df)} results")
print(f"Prototype: {len(prototype_df)} results")
print(f"Gradient-Based: {len(gradient_df)} results")

## Test Scenarios Overview

In [ ]:
# Show unique test scenarios
scenarios = wachter_df[['sample_idx', 'target_idx', 'sample_prediction', 'target_prediction', 'prediction_change']].drop_duplicates()
scenarios['difficulty'] = scenarios['prediction_change'].abs()
scenarios = scenarios.sort_values('difficulty')

print("\nTest Scenarios (sorted by difficulty):")
print("=" * 80)
for _, row in scenarios.iterrows():
    direction = "↑" if row['prediction_change'] > 0 else "↓"
    print(f"Sample {int(row['sample_idx'])} → Target {int(row['target_idx'])}: "
          f"{row['sample_prediction']:.2f} → {row['target_prediction']:.2f} MPG "
          f"({direction} {abs(row['prediction_change']):.2f} MPG)")

---
# 1. Wachter's Method Analysis

**Method**: Optimization-based (L-BFGS-B, SLSQP, Powell)

**Parameters**:
- `lambda`: Weight balancing prediction accuracy vs distance (higher = prioritize target)
- `epsilon`: Tolerance for reaching target prediction

**Key Characteristics**:
- Works with any black-box model
- Non-deterministic (depends on optimizer)
- Can get stuck in local minima
- No training data required

In [ ]:
# Wachter success rate
print("\n" + "=" * 80)
print("WACHTER METHOD - OVERALL STATISTICS")
print("=" * 80)

total_tests = len(wachter_df)
valid_tests = wachter_df['valid'].sum()
success_rate = (valid_tests / total_tests) * 100

print(f"\nTotal tests: {total_tests}")
print(f"Valid counterfactuals: {valid_tests} ({success_rate:.1f}%)")
print(f"Failed: {total_tests - valid_tests} ({100-success_rate:.1f}%)")

if valid_tests > 0:
    valid_wachter = wachter_df[wachter_df['valid'] == True]
    print(f"\nMetrics (valid counterfactuals only):")
    print(f"  Average L2 distance: {valid_wachter['l2_distance'].mean():.4f} ± {valid_wachter['l2_distance'].std():.4f}")
    print(f"  Average sparsity: {valid_wachter['sparsity'].mean():.2f} ± {valid_wachter['sparsity'].std():.2f} features")
    print(f"  Average prediction error: {valid_wachter['prediction_error'].mean():.4f} ± {valid_wachter['prediction_error'].std():.4f}")
    print(f"  Average iterations: {valid_wachter['iterations'].mean():.1f} ± {valid_wachter['iterations'].std():.1f}")
else:
    print("\n⚠️ No valid counterfactuals found!")
    print("Possible reasons:")
    print("  - Epsilon too strict (increase epsilon values)")
    print("  - Lambda not optimal (try different lambda values)")
    print("  - Optimizer stuck in local minima")

In [ ]:
# Wachter: Success rate by scenario
print("\n" + "-" * 80)
print("Success Rate by Scenario")
print("-" * 80)

scenario_success = wachter_df.groupby(['sample_idx', 'target_idx', 'prediction_change']).agg({
    'valid': ['sum', 'count']
}).reset_index()
scenario_success.columns = ['sample_idx', 'target_idx', 'prediction_change', 'valid_count', 'total_count']
scenario_success['success_rate'] = (scenario_success['valid_count'] / scenario_success['total_count']) * 100
scenario_success = scenario_success.sort_values('prediction_change')

for _, row in scenario_success.iterrows():
    print(f"Sample {int(row['sample_idx'])} → Target {int(row['target_idx'])} ({row['prediction_change']:+.2f} MPG): "
          f"{int(row['valid_count'])}/{int(row['total_count'])} valid ({row['success_rate']:.1f}%)")

In [ ]:
# Wachter: Parameter sensitivity
if valid_tests > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Success rate by epsilon
    epsilon_success = wachter_df.groupby('epsilon')['valid'].agg(['sum', 'count'])
    epsilon_success['rate'] = (epsilon_success['sum'] / epsilon_success['count']) * 100
    axes[0].bar(epsilon_success.index, epsilon_success['rate'], color='steelblue', alpha=0.7)
    axes[0].set_xlabel('Epsilon (tolerance)', fontsize=12)
    axes[0].set_ylabel('Success Rate (%)', fontsize=12)
    axes[0].set_title('Wachter: Success Rate by Epsilon', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Success rate by lambda
    lambda_success = wachter_df.groupby('lambda')['valid'].agg(['sum', 'count'])
    lambda_success['rate'] = (lambda_success['sum'] / lambda_success['count']) * 100
    axes[1].bar(lambda_success.index, lambda_success['rate'], color='coral', alpha=0.7)
    axes[1].set_xlabel('Lambda (prediction weight)', fontsize=12)
    axes[1].set_ylabel('Success Rate (%)', fontsize=12)
    axes[1].set_title('Wachter: Success Rate by Lambda', fontsize=14, fontweight='bold')
    axes[1].set_xscale('log')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Best parameters
    print("\n" + "-" * 80)
    print("Best Parameter Combinations")
    print("-" * 80)
    
    valid_wachter = wachter_df[wachter_df['valid'] == True]
    if len(valid_wachter) > 0:
        best_distance = valid_wachter.loc[valid_wachter['l2_distance'].idxmin()]
        print(f"\nClosest counterfactual:")
        print(f"  Lambda={best_distance['lambda']:.2f}, Epsilon={best_distance['epsilon']:.1f}")
        print(f"  L2 Distance={best_distance['l2_distance']:.4f}")
        print(f"  Sparsity={int(best_distance['sparsity'])} features")
        print(f"  Optimizer={best_distance['optimizer']}")
else:
    print("\n⚠️ No valid results to visualize")

---
# 2. Growing Spheres Analysis

**Method**: Geometric search with linear interpolation

**Parameters**:
- `epsilon`: Tolerance for target prediction
- `n_search_samples`: Interpolation granularity (more = finer search)
- `n_top_candidates`: Number of prototypes to try

**Key Characteristics**:
- Requires training data
- Returns synthetic interpolated points (not real instances)
- Deterministic (same data → same result)
- Usually finds closer counterfactuals than Prototype method

In [ ]:
# Growing Spheres success rate
print("\n" + "=" * 80)
print("GROWING SPHERES METHOD - OVERALL STATISTICS")
print("=" * 80)

total_tests = len(growing_spheres_df)
valid_tests = growing_spheres_df['valid'].sum()
success_rate = (valid_tests / total_tests) * 100

print(f"\nTotal tests: {total_tests}")
print(f"Valid counterfactuals: {valid_tests} ({success_rate:.1f}%)")
print(f"Failed: {total_tests - valid_tests} ({100-success_rate:.1f}%)")

if valid_tests > 0:
    valid_gs = growing_spheres_df[growing_spheres_df['valid'] == True]
    print(f"\nMetrics (valid counterfactuals only):")
    print(f"  Average L2 distance: {valid_gs['l2_distance'].mean():.4f} ± {valid_gs['l2_distance'].std():.4f}")
    print(f"  Average sparsity: {valid_gs['sparsity'].mean():.2f} ± {valid_gs['sparsity'].std():.2f} features")
    print(f"  Average prediction error: {valid_gs['prediction_error'].mean():.4f} ± {valid_gs['prediction_error'].std():.4f}")
    print(f"  Average candidates found: {valid_gs['n_candidates_found'].mean():.1f}")
    print(f"  Average candidates tried: {valid_gs['n_candidates_tried'].mean():.1f}")

In [ ]:
# Growing Spheres: Success rate by scenario
print("\n" + "-" * 80)
print("Success Rate by Scenario")
print("-" * 80)

scenario_success = growing_spheres_df.groupby(['sample_idx', 'target_idx', 'prediction_change']).agg({
    'valid': ['sum', 'count']
}).reset_index()
scenario_success.columns = ['sample_idx', 'target_idx', 'prediction_change', 'valid_count', 'total_count']
scenario_success['success_rate'] = (scenario_success['valid_count'] / scenario_success['total_count']) * 100
scenario_success = scenario_success.sort_values('prediction_change')

for _, row in scenario_success.iterrows():
    print(f"Sample {int(row['sample_idx'])} → Target {int(row['target_idx'])} ({row['prediction_change']:+.2f} MPG): "
          f"{int(row['valid_count'])}/{int(row['total_count'])} valid ({row['success_rate']:.1f}%)")

In [ ]:
# Growing Spheres: Parameter effects
if valid_tests > 0:
    valid_gs = growing_spheres_df[growing_spheres_df['valid'] == True]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Success rate by epsilon
    epsilon_success = growing_spheres_df.groupby('epsilon')['valid'].agg(['sum', 'count'])
    epsilon_success['rate'] = (epsilon_success['sum'] / epsilon_success['count']) * 100
    axes[0, 0].bar(epsilon_success.index, epsilon_success['rate'], color='steelblue', alpha=0.7)
    axes[0, 0].set_xlabel('Epsilon (tolerance)', fontsize=11)
    axes[0, 0].set_ylabel('Success Rate (%)', fontsize=11)
    axes[0, 0].set_title('Success Rate by Epsilon', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Distance by n_search_samples
    search_dist = valid_gs.groupby('n_search_samples')['l2_distance'].mean()
    axes[0, 1].plot(search_dist.index, search_dist.values, marker='o', linewidth=2, markersize=8, color='coral')
    axes[0, 1].set_xlabel('n_search_samples', fontsize=11)
    axes[0, 1].set_ylabel('Average L2 Distance', fontsize=11)
    axes[0, 1].set_title('Distance vs Interpolation Granularity', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Distance by n_top_candidates
    cand_dist = valid_gs.groupby('n_top_candidates')['l2_distance'].mean()
    axes[1, 0].plot(cand_dist.index, cand_dist.values, marker='s', linewidth=2, markersize=8, color='green')
    axes[1, 0].set_xlabel('n_top_candidates', fontsize=11)
    axes[1, 0].set_ylabel('Average L2 Distance', fontsize=11)
    axes[1, 0].set_title('Distance vs Number of Prototypes', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Sparsity distribution
    axes[1, 1].hist(valid_gs['sparsity'], bins=range(0, 6), color='purple', alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Sparsity (features changed)', fontsize=11)
    axes[1, 1].set_ylabel('Frequency', fontsize=11)
    axes[1, 1].set_title('Sparsity Distribution', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Best configuration
    print("\n" + "-" * 80)
    print("Best Configuration")
    print("-" * 80)
    
    best_distance = valid_gs.loc[valid_gs['l2_distance'].idxmin()]
    print(f"\nClosest counterfactual:")
    print(f"  Epsilon={best_distance['epsilon']:.1f}, n_search={int(best_distance['n_search_samples'])}, n_candidates={int(best_distance['n_top_candidates'])}")
    print(f"  L2 Distance={best_distance['l2_distance']:.4f}")
    print(f"  Sparsity={int(best_distance['sparsity'])} features")
    print(f"  Candidates found={int(best_distance['n_candidates_found'])}")

---
# 3. Prototype-Based Analysis

**Method**: Nearest real training instance

**Parameters**:
- `epsilon`: Tolerance for target prediction
- `top_k`: Which k-th nearest prototype to return (1=closest, 2=2nd closest, etc.)

**Key Characteristics**:
- Returns REAL training instances (not synthetic)
- Requires training data
- 100% realistic (actually observed)
- Deterministic
- Usually larger distances than Growing Spheres
- Privacy concern: exposes training data

In [ ]:
# Prototype success rate
print("\n" + "=" * 80)
print("PROTOTYPE-BASED METHOD - OVERALL STATISTICS")
print("=" * 80)

total_tests = len(prototype_df)
valid_tests = prototype_df['valid'].sum()
success_rate = (valid_tests / total_tests) * 100

print(f"\nTotal tests: {total_tests}")
print(f"Valid prototypes: {valid_tests} ({success_rate:.1f}%)")
print(f"Failed: {total_tests - valid_tests} ({100-success_rate:.1f}%)")

if valid_tests > 0:
    valid_proto = prototype_df[prototype_df['valid'] == True]
    print(f"\nMetrics (valid prototypes only):")
    print(f"  Average L2 distance: {valid_proto['l2_distance'].mean():.4f} ± {valid_proto['l2_distance'].std():.4f}")
    print(f"  Average sparsity: {valid_proto['sparsity'].mean():.2f} ± {valid_proto['sparsity'].std():.2f} features")
    print(f"  Average prediction error: {valid_proto['prediction_error'].mean():.4f} ± {valid_proto['prediction_error'].std():.4f}")
    print(f"  Average prototypes available: {valid_proto['n_prototypes_available'].mean():.1f}")
    print(f"\n✓ All returned counterfactuals are REAL training instances")

In [ ]:
# Prototype: Success rate by scenario
print("\n" + "-" * 80)
print("Success Rate by Scenario")
print("-" * 80)

scenario_success = prototype_df.groupby(['sample_idx', 'target_idx', 'prediction_change']).agg({
    'valid': ['sum', 'count']
}).reset_index()
scenario_success.columns = ['sample_idx', 'target_idx', 'prediction_change', 'valid_count', 'total_count']
scenario_success['success_rate'] = (scenario_success['valid_count'] / scenario_success['total_count']) * 100
scenario_success = scenario_success.sort_values('prediction_change')

for _, row in scenario_success.iterrows():
    print(f"Sample {int(row['sample_idx'])} → Target {int(row['target_idx'])} ({row['prediction_change']:+.2f} MPG): "
          f"{int(row['valid_count'])}/{int(row['total_count'])} valid ({row['success_rate']:.1f}%)")

In [ ]:
# Prototype: Parameter effects
if valid_tests > 0:
    valid_proto = prototype_df[prototype_df['valid'] == True]
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Success rate by epsilon
    epsilon_success = prototype_df.groupby('epsilon')['valid'].agg(['sum', 'count'])
    epsilon_success['rate'] = (epsilon_success['sum'] / epsilon_success['count']) * 100
    axes[0].bar(epsilon_success.index, epsilon_success['rate'], color='steelblue', alpha=0.7)
    axes[0].set_xlabel('Epsilon (tolerance)', fontsize=12)
    axes[0].set_ylabel('Success Rate (%)', fontsize=12)
    axes[0].set_title('Success Rate by Epsilon', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Distance by top_k
    topk_dist = valid_proto.groupby('top_k')['l2_distance'].mean()
    axes[1].plot(topk_dist.index, topk_dist.values, marker='o', linewidth=2, markersize=8, color='coral')
    axes[1].set_xlabel('top_k (prototype rank)', fontsize=12)
    axes[1].set_ylabel('Average L2 Distance', fontsize=12)
    axes[1].set_title('Distance vs Prototype Rank', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    # Prototypes availability
    epsilon_avail = valid_proto.groupby('epsilon')['n_prototypes_available'].mean()
    axes[2].bar(epsilon_avail.index, epsilon_avail.values, color='green', alpha=0.7)
    axes[2].set_xlabel('Epsilon (tolerance)', fontsize=12)
    axes[2].set_ylabel('Average Prototypes Available', fontsize=12)
    axes[2].set_title('Prototype Availability by Epsilon', fontsize=13, fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Best prototype
    print("\n" + "-" * 80)
    print("Best Prototype")
    print("-" * 80)
    
    best_distance = valid_proto.loc[valid_proto['l2_distance'].idxmin()]
    print(f"\nClosest prototype:")
    print(f"  Epsilon={best_distance['epsilon']:.1f}, top_k={int(best_distance['top_k'])}")
    print(f"  L2 Distance={best_distance['l2_distance']:.4f}")
    print(f"  Sparsity={int(best_distance['sparsity'])} features")
    print(f"  Prototypes available={int(best_distance['n_prototypes_available'])}")
    print(f"  Is real instance: {bool(best_distance['is_real_instance'])}")

---
# 4. Gradient-Based Analysis

**Method**: Neural network gradient descent (TensorFlow)

**Parameters**:
- `learning_rate`: Gradient descent step size
- `lambda`: Weight balancing prediction vs distance
- `epsilon`: Tolerance for target prediction

**Key Characteristics**:
- **ONLY works with neural networks**
- Uses backpropagation for exact gradients
- Fast convergence (typically <100 iterations)
- Non-deterministic
- May produce unrealistic synthetic points
- No training data required

In [ ]:
# Gradient-Based success rate
print("\n" + "=" * 80)
print("GRADIENT-BASED METHOD - OVERALL STATISTICS")
print("=" * 80)

total_tests = len(gradient_df)
valid_tests = gradient_df['valid'].sum()
success_rate = (valid_tests / total_tests) * 100

print(f"\nTotal tests: {total_tests}")
print(f"Valid counterfactuals: {valid_tests} ({success_rate:.1f}%)")
print(f"Failed: {total_tests - valid_tests} ({100-success_rate:.1f}%)")

if valid_tests > 0:
    valid_grad = gradient_df[gradient_df['valid'] == True]
    print(f"\nMetrics (valid counterfactuals only):")
    print(f"  Average L2 distance: {valid_grad['l2_distance'].mean():.4f} ± {valid_grad['l2_distance'].std():.4f}")
    print(f"  Average sparsity: {valid_grad['sparsity'].mean():.2f} ± {valid_grad['sparsity'].std():.2f} features")
    print(f"  Average prediction error: {valid_grad['prediction_error'].mean():.4f} ± {valid_grad['prediction_error'].std():.4f}")
    print(f"  Average iterations: {valid_grad['iterations'].mean():.1f} ± {valid_grad['iterations'].std():.1f}")
else:
    print("\n⚠️ No valid counterfactuals found!")
    print("Possible reasons:")
    print("  - Learning rate too small (slow convergence)")
    print("  - Learning rate too large (instability)")
    print("  - Epsilon too strict (increase epsilon values)")
    print("  - Lambda not optimal (try different values)")

In [ ]:
# Gradient-Based: Success rate by scenario
print("\n" + "-" * 80)
print("Success Rate by Scenario")
print("-" * 80)

scenario_success = gradient_df.groupby(['sample_idx', 'target_idx', 'prediction_change']).agg({
    'valid': ['sum', 'count']
}).reset_index()
scenario_success.columns = ['sample_idx', 'target_idx', 'prediction_change', 'valid_count', 'total_count']
scenario_success['success_rate'] = (scenario_success['valid_count'] / scenario_success['total_count']) * 100
scenario_success = scenario_success.sort_values('prediction_change')

for _, row in scenario_success.iterrows():
    print(f"Sample {int(row['sample_idx'])} → Target {int(row['target_idx'])} ({row['prediction_change']:+.2f} MPG): "
          f"{int(row['valid_count'])}/{int(row['total_count'])} valid ({row['success_rate']:.1f}%)")

In [ ]:
# Gradient-Based: Parameter effects
if valid_tests > 0:
    valid_grad = gradient_df[gradient_df['valid'] == True]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Success rate by epsilon
    epsilon_success = gradient_df.groupby('epsilon')['valid'].agg(['sum', 'count'])
    epsilon_success['rate'] = (epsilon_success['sum'] / epsilon_success['count']) * 100
    axes[0, 0].bar(epsilon_success.index, epsilon_success['rate'], color='steelblue', alpha=0.7)
    axes[0, 0].set_xlabel('Epsilon (tolerance)', fontsize=11)
    axes[0, 0].set_ylabel('Success Rate (%)', fontsize=11)
    axes[0, 0].set_title('Success Rate by Epsilon', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Success rate by learning rate
    lr_success = gradient_df.groupby('learning_rate')['valid'].agg(['sum', 'count'])
    lr_success['rate'] = (lr_success['sum'] / lr_success['count']) * 100
    axes[0, 1].bar(lr_success.index, lr_success['rate'], color='coral', alpha=0.7)
    axes[0, 1].set_xlabel('Learning Rate', fontsize=11)
    axes[0, 1].set_ylabel('Success Rate (%)', fontsize=11)
    axes[0, 1].set_title('Success Rate by Learning Rate', fontsize=12, fontweight='bold')
    axes[0, 1].set_xscale('log')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Success rate by lambda
    lambda_success = gradient_df.groupby('lambda')['valid'].agg(['sum', 'count'])
    lambda_success['rate'] = (lambda_success['sum'] / lambda_success['count']) * 100
    axes[1, 0].bar(lambda_success.index, lambda_success['rate'], color='green', alpha=0.7)
    axes[1, 0].set_xlabel('Lambda (prediction weight)', fontsize=11)
    axes[1, 0].set_ylabel('Success Rate (%)', fontsize=11)
    axes[1, 0].set_title('Success Rate by Lambda', fontsize=12, fontweight='bold')
    axes[1, 0].set_xscale('log')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Iterations distribution
    axes[1, 1].hist(valid_grad['iterations'], bins=20, color='purple', alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Iterations to Convergence', fontsize=11)
    axes[1, 1].set_ylabel('Frequency', fontsize=11)
    axes[1, 1].set_title('Convergence Speed', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Best configuration
    print("\n" + "-" * 80)
    print("Best Configuration")
    print("-" * 80)
    
    best_distance = valid_grad.loc[valid_grad['l2_distance'].idxmin()]
    print(f"\nClosest counterfactual:")
    print(f"  Learning rate={best_distance['learning_rate']:.4f}, Lambda={best_distance['lambda']:.2f}, Epsilon={best_distance['epsilon']:.1f}")
    print(f"  L2 Distance={best_distance['l2_distance']:.4f}")
    print(f"  Sparsity={int(best_distance['sparsity'])} features")
    print(f"  Iterations={int(best_distance['iterations'])}")
else:
    print("\n⚠️ No valid results to visualize")

---
# 5. Comparative Analysis

Compare all four methods across key metrics

In [ ]:
# Overall comparison
print("\n" + "=" * 80)
print("COMPARATIVE SUMMARY - ALL METHODS")
print("=" * 80)

methods_data = []

for name, df in [('Wachter', wachter_df), ('Growing Spheres', growing_spheres_df), 
                  ('Prototype', prototype_df), ('Gradient-Based', gradient_df)]:
    valid = df[df['valid'] == True]
    
    methods_data.append({
        'Method': name,
        'Total Tests': len(df),
        'Valid': len(valid),
        'Success Rate (%)': (len(valid) / len(df)) * 100 if len(df) > 0 else 0,
        'Avg L2 Distance': valid['l2_distance'].mean() if len(valid) > 0 else np.nan,
        'Avg Sparsity': valid['sparsity'].mean() if len(valid) > 0 else np.nan,
        'Avg Pred Error': valid['prediction_error'].mean() if len(valid) > 0 else np.nan
    })

comparison_df = pd.DataFrame(methods_data)
print("\n", comparison_df.to_string(index=False))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Success rates
axes[0, 0].bar(comparison_df['Method'], comparison_df['Success Rate (%)'], 
               color=['steelblue', 'coral', 'green', 'purple'], alpha=0.7)
axes[0, 0].set_ylabel('Success Rate (%)', fontsize=12)
axes[0, 0].set_title('Success Rate Comparison', fontsize=13, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=15)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Average distances (only valid)
valid_methods = comparison_df[~comparison_df['Avg L2 Distance'].isna()]
if len(valid_methods) > 0:
    axes[0, 1].bar(valid_methods['Method'], valid_methods['Avg L2 Distance'], 
                   color=['steelblue', 'coral', 'green', 'purple'][:len(valid_methods)], alpha=0.7)
    axes[0, 1].set_ylabel('Average L2 Distance', fontsize=12)
    axes[0, 1].set_title('Distance Comparison (valid only)', fontsize=13, fontweight='bold')
    axes[0, 1].tick_params(axis='x', rotation=15)
    axes[0, 1].grid(True, alpha=0.3, axis='y')
else:
    axes[0, 1].text(0.5, 0.5, 'No valid results', ha='center', va='center', fontsize=14)
    axes[0, 1].set_title('Distance Comparison', fontsize=13, fontweight='bold')

# Average sparsity
if len(valid_methods) > 0:
    axes[1, 0].bar(valid_methods['Method'], valid_methods['Avg Sparsity'], 
                   color=['steelblue', 'coral', 'green', 'purple'][:len(valid_methods)], alpha=0.7)
    axes[1, 0].set_ylabel('Average Sparsity (features)', fontsize=12)
    axes[1, 0].set_title('Sparsity Comparison (valid only)', fontsize=13, fontweight='bold')
    axes[1, 0].tick_params(axis='x', rotation=15)
    axes[1, 0].grid(True, alpha=0.3, axis='y')
else:
    axes[1, 0].text(0.5, 0.5, 'No valid results', ha='center', va='center', fontsize=14)
    axes[1, 0].set_title('Sparsity Comparison', fontsize=13, fontweight='bold')

# Prediction error
if len(valid_methods) > 0:
    axes[1, 1].bar(valid_methods['Method'], valid_methods['Avg Pred Error'], 
                   color=['steelblue', 'coral', 'green', 'purple'][:len(valid_methods)], alpha=0.7)
    axes[1, 1].set_ylabel('Average Prediction Error', fontsize=12)
    axes[1, 1].set_title('Prediction Accuracy (valid only)', fontsize=13, fontweight='bold')
    axes[1, 1].tick_params(axis='x', rotation=15)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
else:
    axes[1, 1].text(0.5, 0.5, 'No valid results', ha='center', va='center', fontsize=14)
    axes[1, 1].set_title('Prediction Accuracy', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## Key Insights

### Growing Spheres vs Prototype:
- **Same success rate** (both require training data, fail when epsilon too small)
- **Same distances** when both valid
- **Difference**: Growing Spheres returns synthetic interpolated points, Prototype returns real instances

### Wachter vs Gradient-Based:
- Both optimization methods (no training data needed)
- Wachter: Uses numerical optimization (scipy), works with any model
- Gradient-Based: Uses backpropagation, **only works with neural networks**
- Both sensitive to parameter tuning (lambda, epsilon, learning rate)

### Method Selection Guide:
1. **Have training data + Want realistic**: Use **Prototype**
2. **Have training data + Want closer**: Use **Growing Spheres**
3. **No training data + Neural network**: Use **Gradient-Based**
4. **No training data + Any model**: Use **Wachter**

---
# 6. Recommendations

Based on this analysis, here are recommendations for improving results:

In [ ]:
print("\n" + "=" * 80)
print("RECOMMENDATIONS FOR IMPROVING RESULTS")
print("=" * 80)

# Check which methods failed
wachter_success_rate = (wachter_df['valid'].sum() / len(wachter_df)) * 100
gradient_success_rate = (gradient_df['valid'].sum() / len(gradient_df)) * 100
gs_success_rate = (growing_spheres_df['valid'].sum() / len(growing_spheres_df)) * 100
proto_success_rate = (prototype_df['valid'].sum() / len(prototype_df)) * 100

print("\n1. WACHTER METHOD:")
if wachter_success_rate < 50:
    print("   ⚠️ Low success rate - Try these improvements:")
    print("   • Increase epsilon values: [2.0, 5.0, 10.0, 15.0]")
    print("   • Test more lambda values: [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]")
    print("   • Increase max_iter: 1500-2000")
    print("   • Current: Only testing 1 configuration per scenario")
else:
    print("   ✓ Good success rate - Consider fine-tuning to reduce distances")

print("\n2. GROWING SPHERES:")
if gs_success_rate < 50:
    print("   ⚠️ Low success rate - Try these improvements:")
    print("   • Increase epsilon values: [2.0, 5.0, 10.0, 15.0]")
    print("   • More prototypes available = higher success rate")
else:
    print("   ✓ Good success rate")
    print("   • To reduce distances: Increase n_search_samples (50-100)")
    print("   • To explore more: Increase n_top_candidates (15-20)")

print("\n3. PROTOTYPE-BASED:")
if proto_success_rate < 50:
    print("   ⚠️ Low success rate - Try these improvements:")
    print("   • Increase epsilon values: [2.0, 5.0, 10.0, 15.0]")
    print("   • Limited by training data coverage")
else:
    print("   ✓ Good success rate")
    print("   • Returns REAL training instances (most interpretable)")
    print("   • Distances typically same as Growing Spheres at α=1.0")

print("\n4. GRADIENT-BASED:")
if gradient_success_rate < 50:
    print("   ⚠️ Low success rate - Try these improvements:")
    print("   • Increase epsilon values: [2.0, 5.0, 10.0, 15.0]")
    print("   • Test learning_rate: [0.005, 0.01, 0.05, 0.1]")
    print("   • Test more lambda values: [0.01, 0.1, 0.5, 1.0, 5.0]")
    print("   • Increase max_iter: 800-1000")
else:
    print("   ✓ Good success rate - Fast convergence with gradient descent")

print("\n" + "=" * 80)
print("NEXT STEPS:")
print("=" * 80)
print("\n1. Update stand_meth_test_manager.py with recommended parameter ranges")
print("2. Re-run tests: python stand_meth_test_manager.py")
print("3. Re-run this analysis notebook to compare results")
print("4. Once satisfied, use best parameters in main experiments")
print("\n" + "=" * 80)